# UABR-SLM · Notebook 12 — New methods: decision-slot anchoring during SFT (A1, A2) and readout repair after SFT (B)

GPU **T4** (×1) · ~1.5 h (6 SFT runs of ~9 min + short repairs + evaluation) · validation only (test sealed).
Safe to re-run with this notebook's previous output attached: finished runs are reused.

**Why (notebook 11):** after plain SFT a re-fitted linear probe still reads the routing decision (0.788 → 0.79) while the
pre-SFT probe falls to chance and the token readout drops (0.838 → 0.70): the decision information *moves* and the
`<ANSWER>`/`<CALL>` readout does not follow. Interpolation (α = 0.5) and the routing task vector repair it without training.

**Design (fixed before results)**

| | Method | Labels? | Detail |
|---|---|---|---|
| 0 | **Check: cross-validated probe** | val labels (CV) | 5-fold GroupKFold (subject groups) logistic probe on the *validation* hidden states at `<ASSISTANT>`, E0 and E1 s11/s29. Recovered share with this ceiling, same rule as notebook 11 (≥ 0.8 readout, ≤ 0.2 representation) |
| A1 | **SFT-prompt anchoring** | none, no extra data | during the 07 SFT recipe (same batches, seeds 11/29), add γ · softBCE(σ(d_student), σ(d_teacher)) at the `<ASSISTANT>` position of every SFT example; d = z_CALL − z_ANSWER, teacher = frozen θ_route |
| A2 | **Question-pool anchoring** | none (questions only) | same loss on 16 random half-B question prompts per update (teacher scores precomputed) |
| A2-critic | A2 for the notebook 10 critic | none | anchors both slots: routing (half-B questions) and IsRel (half-B windows, pair `<IRR>`/`<REL>`), 16 + 16 prompts per update |
| B | **Readout repair after SFT** | half-B labels | from the 07 SFT model: only the `<ANSWER>`/`<CALL>` embedding rows + final RMSNorm gain train (2-logit CE) + SFT replay; best selection-subset AUROC among checkpoints whose instruction val loss ≤ E1 + 0.02 |
| ref | E0, E1, interpolation α = 0.5, task vector β = 1 | — | rebuilt and scored on the same questions |

γ = 1 (fixed, not tuned). Selection subset = 300 validation questions (used only by B); all routing results are reported on
the **other** 1,454 validation questions (EM on 500 of them). Primary: routing AUROC vs frozen θ0 labels; paired clustered bootstrap vs E1, interpolation α = 0.5 and the task vector, per seed.

**Inputs:** outputs of notebooks **01, 02, 05, 06 (v2), 07, 10** (optional: **08**, **11** for reference numbers).

## 1 · Get the project code

In [ ]:
import os, sys, glob, json, subprocess, platform, time, shutil
ON_KAGGLE = os.path.exists("/kaggle/working")

REPO_URL = "https://github.com/Prajwal-13-dev/uabr_slm.git"
REPO_REF = "main"        # pin to a commit hash for final, reportable runs

def find_code_root():
    if os.environ.get("UABR_CODE"):
        return os.path.abspath(os.environ["UABR_CODE"])
    for base in glob.glob("/kaggle/input/*"):
        for root, dirs, _ in os.walk(base):
            if os.path.isdir(os.path.join(root, "src", "uabr")):
                return root
            if root.count(os.sep) - base.count(os.sep) >= 4:
                dirs[:] = []
    dest = "/tmp/uabr_slm"
    if not os.path.isdir(dest):
        url = REPO_URL
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
            url = url.replace("https://", f"https://{tok}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "-q", url, dest], check=True)
        subprocess.run(["git", "-C", dest, "checkout", "-q", REPO_REF], check=True)
    return dest

CODE_ROOT = find_code_root()
SRC = os.path.join(CODE_ROOT, "src")
sys.path.insert(0, SRC)
CODE_COMMIT = subprocess.run(["git", "-C", CODE_ROOT, "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip() or "unknown"

def find_input(pattern, env=None):
    """Locate a file from an attached notebook output (or an env override for local tests)."""
    if env and os.environ.get(env):
        return os.environ[env]
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"{pattern} not found under /kaggle/input - attach the required notebook output (Add Input → Your Work)"
    return hits[0]

T0 = time.time()
def stage(msg): print(f"[{(time.time()-T0)/60:6.1f} min] {msg}", flush=True)
print("code:", CODE_ROOT, "| commit:", CODE_COMMIT[:10])

import torch
from sklearn.model_selection import GroupKFold
from uabr.data import tokenizer as T
from uabr.data import qa_format as Q
from uabr.eval import prompts as PR
from uabr.eval.batched import greedy_batch
from uabr.eval.generate import load_model
from uabr.eval.metrics import alias_em
from uabr.retrieval.select import select_non_overlapping
from uabr.routing.metrics import auroc, clustered_bootstrap_auroc
from uabr.routing.pilot import paired_cluster_bootstrap
from uabr.routing.router import router_prompt, router_scores
from uabr.critic.selfrag import critique_scores, last_hidden
from uabr.critic.windows import window_items
from uabr.analysis.readout import fit_probe, probe_scores, interpolate, task_vector_add
from uabr.train.sft import STANDARD, instruction_batches, train_sft, sft_val_loss
from uabr.train.anchor import SFTSlotAnchor, PoolAnchor, repair_readout

## 2 · Configuration and inputs

In [ ]:
import re
import numpy as np, pandas as pd
OUT = "/kaggle/working/12_methods" if ON_KAGGLE else os.path.abspath("outputs/12_methods")
os.makedirs(OUT, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
if ON_KAGGLE: assert DEV == "cuda", "No GPU - set Accelerator to GPU T4"
SEEDS = [int(s) for s in os.environ.get("SEEDS", "11,29").split(",")]
GAMMA, ANCHOR_BS = 1.0, 16
N_BOOT = int(os.environ.get("N_BOOT", 1000)); SEED = 1212
SEL_N, PRES_N = int(os.environ.get("SEL_N", 300)), int(os.environ.get("PRES_N", 500))
R_LR, R_EPOCHS, R_EVAL = 1e-3, int(os.environ.get("R_EPOCHS", 3)), int(os.environ.get("R_EVAL", 25))
MAX_NEW, SEQ = 16, 512

def one_input(pattern, env=None):
    if env and os.environ.get(env): return os.environ[env]
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert len(hits) == 1, f"{pattern}: expected exactly 1 attached file, found {len(hits)}: {hits}"
    return hits[0]
def optional(pattern, env=None):
    if env and os.environ.get(env): return os.environ[env]
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)); return hits[0] if len(hits) == 1 else None

tok = T.load(find_input("01_corpus/tokenizer/tokenizer.json", "TOKENIZER"))
top = pd.read_parquet(find_input("02_bm25/retrieval/popqa_top20.parquet", "TOP20")).set_index("id")
wins = pd.read_parquet(find_input("02_bm25/retrieval/windows_retrieved.parquet", "WINDOWS"))
THETA0 = one_input("05_mixed/arm_M/model_final.pt", "THETA0")
POLICY_PATH = one_input("06_routing/routing_policy.json", "POLICY"); D06 = os.path.dirname(POLICY_PATH)
ROUTER = os.path.join(D06, "router_action.pt"); LAM = float(json.load(open(POLICY_PATH))["lambda"])
V = pd.read_parquet(os.path.join(D06, "labels_val.parquet")).reset_index(drop=True)
B = pd.read_parquet(os.path.join(D06, "labels_halfB.parquet")).reset_index(drop=True)
D07 = os.path.dirname(one_input("07_sft_pilot/pilot_report.json", "REPORT07"))
D10 = os.path.dirname(one_input("10_selfrag/critic.pt", "CRITIC")); CRITIC = os.path.join(D10, "critic.pt")
R08, R11 = optional("08_replay/results_val.csv", "R08"), optional("11_posthoc/interpolation.csv", "R11")
E1 = {s: os.path.join(D07, f"sft_s{s}", "final.pt") for s in SEEDS}; C1 = {s: os.path.join(D10, f"sft_s{s}", "final.pt") for s in SEEDS}
for p in list(E1.values()) + list(C1.values()): assert os.path.exists(p), p
WTEXT = dict(zip(wins["doc_id"], wins["text"]))
WART, WSTART = dict(zip(wins["doc_id"], wins["article_idx"])), dict(zip(wins["doc_id"], wins["start_word"]))
stop_ids = [i for t, i in tok.get_vocab().items() if "\n" in tok.decode([i])]
sx = pd.read_parquet(os.path.join(D07, "sft_examples.parquet"))
mk = lambda r: (list(map(int, r.tokens)), [False] * int(r.n_prompt) + [True] * (len(r.tokens) - int(r.n_prompt)))
SFT_TRAIN = [mk(r) for r in sx[sx.part == "train"].itertuples(index=False)]
SFT_VALX = [mk(r) for r in sx[sx.part == "val"].itertuples(index=False)]
SELQ = V.sample(min(SEL_N, len(V) // 3), random_state=SEED)
EV = V[~V.id.isin(SELQ.id)].reset_index(drop=True)
PRESQ = EV.sample(min(PRES_N, len(EV)), random_state=SEED + 1)
RP_B = [router_prompt(tok, q, "action") for q in B.question]; yB = B.y.astype(int).tolist()
RP_EV = [router_prompt(tok, q, "action") for q in EV.question]; RP_SEL = [router_prompt(tok, q, "action") for q in SELQ.question]
RP_V = [router_prompt(tok, q, "action") for q in V.question]
G2W = pd.read_parquet(os.path.join(D10, "g2_windows.parquet")); EVW = pd.read_parquet(os.path.join(D10, "eval_windows.parquet"))
for d in (G2W, EVW): d["prompt"] = [Q.evidence_prompt_ids(tok, q, [WTEXT[x]], SEQ - MAX_NEW)[0] for q, x in zip(d.question, d.doc_id)]
print(f"val {len(V):,} -> selection {len(SELQ)} (B only) + evaluation {len(EV):,} (EM subset {len(PRESQ)}) | half-B {len(B):,} | "
      f"SFT {len(SFT_TRAIN):,} examples | G2 windows {len(G2W)} | val windows {len(EVW):,} | refs 08 {bool(R08)} 11 {bool(R11)}")

## 3 · Check: cross-validated probe ceiling on the validation questions' own hidden states

In [ ]:
yV, gV = V.y.astype(int).values, V.group_id.values
def cv_probe(h):
    oof = np.zeros(len(yV))
    for tr, te in GroupKFold(5).split(h, yV, gV):
        oof[te] = probe_scores(fit_probe(h[tr], yV[tr]), h[te])
    return auroc(yV, oof)
CV = []
for name, path in [("E0", ROUTER)] + [(f"E1_s{s}", E1[s]) for s in SEEDS]:
    m = load_model(path, DEV); h = last_hidden(m, RP_V, DEV); tokv = auroc(yV, router_scores(m, RP_V, "action", DEV)[0]); del m
    CV.append({"model": name, "token": tokv, "cv_probe": cv_probe(h)})
CV = pd.DataFrame(CV); t0 = CV.loc[CV.model == "E0", "token"].item(); c0 = CV.loc[CV.model == "E0", "cv_probe"].item()
CV["recovered_share"] = [(r.cv_probe - r.token) / (t0 - r.token) if t0 - r.token >= 0.02 else np.nan for r in CV.itertuples()]
CV["cv_probe_drop_vs_E0"] = c0 - CV.cv_probe
CV["reading"] = ["" if s != s else ("readout-dominated" if s >= 0.8 else ("representation-dominated" if s <= 0.2 else "mixed")) for s in CV.recovered_share]
display(CV.round(4)); CV.to_csv(f"{OUT}/cv_probe_check.csv", index=False)

## 4 · Method A1 / A2 (routing) and A2-critic: anchored SFT runs

In [ ]:
T_START = time.time(); RUNS = {}
def run(name, init, seed, aux_fn):
    out = f"{OUT}/{name}"
    for cand in [out] + [os.path.dirname(p) for p in sorted(glob.glob(f"/kaggle/input/**/12_methods/{name}/summary.json", recursive=True))]:
        if os.path.exists(os.path.join(cand, "final.pt")) and os.path.exists(os.path.join(cand, "summary.json")):
            RUNS[name] = json.load(open(os.path.join(cand, "summary.json"))); stage(f"{name}: reusing {cand}"); return os.path.join(cand, "final.pt")
    m = load_model(init, DEV); aux = aux_fn()
    stage(f"{name}: init {os.path.basename(os.path.dirname(init)) or init} | seed {seed} | gamma {GAMMA}")
    s = train_sft(m, instruction_batches(SFT_TRAIN, STANDARD["batch_size"], seed), dict(STANDARD, log_every_steps=250), seed, out,
                  log=lambda msg: stage(f"  [{name}] {msg.strip()}"), aux=aux, gamma=GAMMA)
    RUNS[name] = s; del m, aux
    if DEV == "cuda": torch.cuda.empty_cache()
    stage(f"{name}: done in {s['minutes']} min"); return f"{out}/final.pt"

m0 = load_model(ROUTER, DEV); TD_B = router_scores(m0, RP_B, "action", DEV)[0]; del m0
mc = load_model(CRITIC, DEV); BW = window_items(B, top, tok, WTEXT, WART, WSTART, 10, SEQ - MAX_NEW)
REL, IRR = mc.cfg.vocab_size - 2, mc.cfg.vocab_size - 1
TDC_B = router_scores(mc, RP_B, "action", DEV)[0]; TDC_W = critique_scores(mc, BW.prompt.tolist(), DEV); del mc
print(f"teacher scores: theta_route on {len(RP_B):,} half-B questions | critic on {len(RP_B):,} questions + {len(BW):,} half-B windows")
CK = {}
for seed in SEEDS:
    CK[("A1", seed)] = run(f"A1_sftslot_s{seed}", ROUTER, seed, lambda: SFTSlotAnchor(load_model(ROUTER, DEV)))
    CK[("A2", seed)] = run(f"A2_pool_s{seed}", ROUTER, seed, lambda: PoolAnchor([(RP_B, TD_B, (T.ANSWER_ID, T.CALL_ID))], bs=ANCHOR_BS, seed=seed))
    CK[("A2c", seed)] = run(f"A2_critic_s{seed}", CRITIC, seed, lambda: PoolAnchor(
        [(RP_B, TDC_B, (T.ANSWER_ID, T.CALL_ID)), (BW.prompt.tolist(), TDC_W, (IRR, REL))], bs=ANCHOR_BS, seed=seed))

## 5 · Method B: readout repair of the 07 SFT models

In [ ]:
def closed_em(m, qdf, bans=(T.CALL_ID,)):
    g = greedy_batch(m, [Q.closed_prompt_ids(tok, q) for q in qdf.question], DEV, MAX_NEW, banned=bans, stop_ids=stop_ids)
    return float(np.mean([alias_em(PR.clean_generation(tok.decode(x)), a) for x, a in zip(g, qdf.gold_aliases)]))
def evidence_em(m, qdf, bans=(T.CALL_ID,)):
    P = []
    for r in qdf.itertuples(index=False):
        sel, _ = select_non_overlapping(list(top.loc[r.id].doc_ids), WART, WSTART, k=3, window_words=60)
        P.append(Q.evidence_prompt_ids(tok, r.question, [WTEXT[d] for d in sel], SEQ - MAX_NEW)[0])
    g = greedy_batch(m, P, DEV, MAX_NEW, banned=bans, stop_ids=stop_ids)
    return float(np.mean([alias_em(PR.clean_generation(tok.decode(x)), a) for x, a in zip(g, qdf.gold_aliases)]))
REPAIR = {}
for seed in SEEDS:
    out = f"{OUT}/B_repair_s{seed}"; os.makedirs(out, exist_ok=True)
    prev = sorted(glob.glob(f"/kaggle/input/**/12_methods/B_repair_s{seed}/repaired.pt", recursive=True))
    if os.path.exists(f"{out}/repaired.pt") or prev:
        CK[("B", seed)] = f"{out}/repaired.pt" if os.path.exists(f"{out}/repaired.pt") else prev[0]; stage(f"B seed {seed}: reusing"); continue
    m = load_model(E1[seed], DEV); L1 = sft_val_loss(m, SFT_VALX, DEV)
    def ev(mm, L1=L1):
        a = auroc(SELQ.y, router_scores(mm, RP_SEL, "action", DEV)[0]); l = sft_val_loss(mm, SFT_VALX, DEV)
        return a, l <= L1 + 0.02, {"sel_auroc": round(a, 4), "instr_val_loss": round(l, 4)}
    stage(f"B seed {seed}: repairing E1 (instruction val loss {L1:.3f})")
    best, hist = repair_readout(m, RP_B, yB, DEV, sft_replay=SFT_TRAIN, lr=R_LR, epochs=R_EPOCHS, eval_every=R_EVAL, seed=seed, evaluate=ev, log=stage)
    m.load_state_dict(best["state"])
    torch.save({"model": best["state"], "model_cfg": m.cfg.to_dict(), "best_step": best["step"], "info": best["info"]}, f"{out}/repaired.pt")
    pd.DataFrame(hist).to_csv(f"{out}/history.csv", index=False)
    REPAIR[seed] = {"best_step": best["step"], "info": best["info"]}; CK[("B", seed)] = f"{out}/repaired.pt"; del m
    stage(f"B seed {seed}: best step {best['step']} {best['info']}")

## 6 · References rebuilt on the same questions: interpolation α = 0.5 and task vector β = 1

In [ ]:
sd = lambda p: torch.load(p, map_location="cpu", weights_only=False)["model"]
SD_R, SD_T0 = sd(ROUTER), sd(THETA0)
for seed in SEEDS:
    SD_S = sd(E1[seed])
    for name, st in [("interp0.5", interpolate(SD_R, SD_S, 0.5)), ("taskvec1", task_vector_add(SD_S, SD_R, SD_T0, 1.0))]:
        p = f"{OUT}/ref_{name}_s{seed}.pt"
        if not os.path.exists(p): torch.save({"model": st, "model_cfg": load_model(ROUTER, "cpu").cfg.to_dict()}, p)
        CK[(name, seed)] = p
CK[("E0", "all")] = ROUTER
for s in SEEDS: CK[("E1", s)] = E1[s]
print(sorted(CK.keys(), key=str))

## 7 · Evaluate routing models (evaluation questions) and critic models (IsRel)

In [ ]:
yE, gE = EV.y.astype(int).values, EV.group_id.values
S = {}; ROWS = []
for (meth, seed), path in sorted(CK.items(), key=lambda kv: str(kv[0])):
    if meth == "A2c": continue
    m = load_model(path, DEV)
    s = router_scores(m, RP_EV, "action", DEV)[0]; S[(meth, seed)] = s
    ci = clustered_bootstrap_auroc(yE, s, gE, n_boot=N_BOOT)
    ROWS.append({"method": meth, "seed": seed, "auroc_eval": ci["auroc"], "lo": ci["lo"], "hi": ci["hi"],
                 "auroc_full_val": auroc(yV, router_scores(m, RP_V, "action", DEV)[0]), "default_call": float((s > 0).mean()),
                 "instr_val_loss": sft_val_loss(m, SFT_VALX, DEV), "closed_em": closed_em(m, PRESQ), "evidence_em": evidence_em(m, PRESQ)})
    del m
    r = ROWS[-1]; stage(f"{meth} s{seed}: AUROC {r['auroc_eval']:.3f} [{r['lo']:.3f}, {r['hi']:.3f}] | call {r['default_call']:.1%} | instr loss {r['instr_val_loss']:.3f} | closed {r['closed_em']:.1%} | evidence {r['evidence_em']:.1%}")
D0 = S[("E0", "all")] > 0
RES = pd.DataFrame(ROWS); RES["agree_E0"] = [float(((S[(m, s)] > 0) == D0).mean()) for m, s in zip(RES.method, RES.seed)]
L0 = RES.loc[RES.method == "E0", "instr_val_loss"].item()
RES["sft_gain_kept"] = [(L0 - l) / (L0 - RES[(RES.method == "E1") & (RES.seed == s)].instr_val_loss.item()) if s != "all" else 0.0
                        for l, s in zip(RES.instr_val_loss, RES.seed)]
RES.to_csv(f"{OUT}/results_routing.csv", index=False)
pd.set_option("display.width", 250); display(RES.round(4))
print("mean over seeds:"); display(RES[RES.seed != "all"].groupby("method")[["auroc_eval", "default_call", "agree_E0", "instr_val_loss", "sft_gain_kept", "closed_em", "evidence_em"]].mean().round(4))

CROWS = []
for (name, seed), path in [(("critic", "all"), CRITIC)] + [(("critic_SFT", s), C1[s]) for s in SEEDS] + [(("A2c", s), CK[("A2c", s)]) for s in SEEDS]:
    m = load_model(path, DEV)
    CROWS.append({"model": name, "seed": seed, "isrel_g2": auroc(G2W.g2_label, critique_scores(m, G2W.prompt.tolist(), DEV)),
                  "isrel_val": auroc(EVW.auto_rel, critique_scores(m, EVW.prompt.tolist(), DEV)),
                  "route_auroc_eval": auroc(yE, router_scores(m, RP_EV, "action", DEV)[0]), "instr_val_loss": sft_val_loss(m, SFT_VALX, DEV)})
    del m
CRES = pd.DataFrame(CROWS); CRES.to_csv(f"{OUT}/results_critic.csv", index=False); display(CRES.round(4))

## 8 · Paired comparisons (clustered bootstrap, evaluation questions)

In [ ]:
CMP = []
for seed in SEEDS:
    for a in ["A1", "A2", "B"]:
        for b in ["E1", "interp0.5", "taskvec1"]:
            f = lambda idx, w, a=a, b=b: auroc(yE[idx], S[(a if w == "a" else b, seed)][idx])
            r = paired_cluster_bootstrap(f, gE, n_boot=N_BOOT, seed=seed); CMP.append({"seed": seed, "comparison": f"{a} - {b}", "metric": "routing AUROC", **r})
    for lab, d, col in [("IsRel G2", G2W, "g2_label"), ("IsRel val", EVW, "auto_rel")]:
        m1, m2 = load_model(CK[("A2c", seed)], DEV), load_model(C1[seed], DEV)
        s1, s2 = critique_scores(m1, d.prompt.tolist(), DEV), critique_scores(m2, d.prompt.tolist(), DEV); del m1, m2
        f = lambda idx, w: auroc(d[col].values[idx], (s1 if w == "a" else s2)[idx])
        r = paired_cluster_bootstrap(f, d.group_id.values, n_boot=N_BOOT, seed=seed); CMP.append({"seed": seed, "comparison": "A2c - critic_SFT", "metric": lab, **r})
CMP = pd.DataFrame(CMP); CMP["excludes_0"] = (CMP.ci_lo > 0) | (CMP.ci_hi < 0); CMP.to_csv(f"{OUT}/comparisons.csv", index=False)
display(CMP.drop(columns="n_boot_valid").round(4))
if R08:
    r8 = pd.read_csv(R08); print("reference, notebook 08 (full validation AUROC):"); display(r8[["arm", "seed", "auroc", "default_call_rate", "instr_val_loss"]].round(4))

## 9 · Save + checks

In [ ]:
checks = []
def add(name, ok, detail, severity="fail"):
    checks.append({"name": name, "passed": bool(ok), "detail": detail, "severity": severity})
p07 = pd.read_csv(os.path.join(D07, "pilot_table.csv")); p07 = p07[p07.frac == 1.0]; ref07 = {int(r.seed): r.auroc for r in p07.itertuples()}
e1 = {s: RES[(RES.method == "E1") & (RES.seed == s)].auroc_full_val.item() for s in SEEDS}
add("E1_matches_07", all(abs(e1[s] - ref07[s]) < 1e-3 for s in SEEDS), f"{e1} vs {ref07}")
add("E0_matches_06", abs(RES.loc[RES.method == "E0", "auroc_full_val"].item() - auroc(yV, V.router_score)) < 1e-3, "E0 full-val AUROC")
ups = {s["updates"] for s in RUNS.values()}
add("anchored_runs_equal_updates", len(ups) == 1, f"updates {ups}")
add("B_qualified", all(not v["info"].get("unqualified", False) for v in REPAIR.values()), f"{REPAIR}", "warn")
c10 = CRES.loc[CRES.model == "critic", "isrel_g2"].item()
add("critic_matches_10", abs(c10 - 0.8719) < 2e-3, f"critic IsRel G2 {c10:.4f}", "warn")

fails = [c for c in checks if not c["passed"] and c["severity"] == "fail"]
warns = [c for c in checks if not c["passed"] and c["severity"] == "warn"]
status = "FAIL" if fails else ("PASS_WITH_WARNINGS" if warns else "PASS")
for c in checks:
    print(f"[{'PASS' if c['passed'] else c['severity'].upper():4}] {c['name']:<38} {c['detail']}")

json.dump({"notebook": "12_methods", "status": status, "checks": checks, "gamma": GAMMA, "repair": REPAIR, "runs": RUNS,
           "cv_check": CV.to_dict("records"), "code": {"repo": REPO_URL, "ref": REPO_REF, "commit": CODE_COMMIT}},
          open(f"{OUT}/report.json", "w"), indent=1, default=float)
print(f"\nNOTEBOOK 12: {status}")